# Receiver query-adapter pilot (twelve carriers), Kaggle T4

Question: can B learn to use the unchanged transferred result states, without rebuilding them or receiving attention from a reference model? This applies Attuner's query-side repair (LoRA on q_proj, frozen cache, no reference at inference) to the selected-result setting; it is not a new method. The earlier fresh-key result (63 -> 94 labels) motivates it but does not show an adapter can reproduce it.

## Now: step 4, untouched eval families (run cells 1-3, then 9-13; skip 4-7)

The dev gates all passed (adapted 144 vs unadapted 94 of 144, no-R 83, mismatched follows partner 57 vs own 5, A-change pairs 12 vs 3, malformed 0 vs 0). Epoch 1 was selected by the tie rule.

Upload the downloaded `results_stage_plan4_qadapt/` folder from the training run (it must contain `adapter_epoch1.pt`, `selection.json`, `dev_summary.json`, `train_summary.json`, `smoke_check.json`) next to `src/`. Do **not** retrain: a retrained adapter would be a different candidate.

- **Cell 9** freezes the candidate: hashes `adapter_epoch1.pt` into `frozen_candidate.json` before any eval sheet exists.
- **Cell 10** writes A's sheets for the 24 eval families (24 base + 24 a_flip) with no adapter attached (~50 min; resumes).
- **Cell 11** checks construction (templates match the frozen ones, support held, mismatched/no-R caches) and A's written accuracy.
- **Cell 12** runs every condition on one sheet so the rows can be inspected.
- **Cell 13** runs the rest once with the frozen checkpoint (~60 min; resumes) and prints labels, the message-dependence checks, paired comparisons and costs.

Conditions: adapted / unadapted transferred R (base + a_flip), adapted / unadapted no R and mismatched R (base), reconstruction fk_fv, full KV and result text (base + a_flip). Reported whatever they show.

---

## Earlier: pilot cells

**Cell 4** reruns the implementation check. **Cell 5** trains (96 sheets, 3 epochs). **Cell 6** selects the checkpoint on dev base. **Cell 7** runs the dev gates.

---

## Earlier: implementation check

This notebook first ran **only step 1**, the implementation check on the first 4 training base sheets (3 optimizer steps): zero-initialised reproduction, gradients on adapter tensors only, finite loss, unchanged received caches, teacher plan absent from the plan input, kernel agreement, and reset afterwards.

Upload `src/`, `configs/`, `scripts/`, `data/plan4_qadapt_train.jsonl`, `data/plan4_qadapt_dev.jsonl`, `data/plan4_qadapt_eval.jsonl`, `data/plan4_timing12.jsonl`, and the folder `results_stage_plan4_qadapt_prep/`.

Download `/kaggle/working/results_stage_plan4_qadapt`.


In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
STAGE = "stage_plan4_qadapt"

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if f"{STAGE}.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / f"{STAGE}.py").is_file() and (cand / "data" / "plan4_qadapt_train.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError(f"Need src/{STAGE}.py and data/plan4_qadapt_train.jsonl.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input") or not (dest / "src" / f"{STAGE}.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO", REPO.resolve())
prep = REPO / "results_stage_plan4_qadapt_prep"
for name in ("a_artifacts_train.jsonl", "a_artifacts_dev.jsonl", "teacher_train.jsonl", "check_summary.json", "timing12_a_artifacts_12.jsonl"):
    print("prep", name, (prep / name).is_file())
for name in ("plan4_qadapt_train.jsonl", "plan4_qadapt_dev.jsonl", "plan4_qadapt_eval.jsonl", "plan4_timing12.jsonl"):
    print("data", name, (REPO / "data" / name).is_file())

def out_file(name):
    for d in (WORK / "results_stage_plan4_qadapt", REPO / "results" / STAGE / "run"):
        if (d / name).is_file():
            return d / name
    raise FileNotFoundError(name)

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu, "| torch", torch.__version__)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
print("Dest /kaggle/working/results_stage_plan4_qadapt. Implementation check only.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", f"src.{STAGE}", "--phase", "smoke"])
print("smoke exit", rc)
r = json.loads(out_file("smoke_check.json").read_text(encoding="utf-8"))
print("adapter", r["adapter"])
print("\nsheets")
for s in r["sheets"]:
    c = s["checks"]
    print(s["example_id"], "targets_ok", c["ok"], "| plan not in plan input", c["plan_not_in_plan_input"],
          "| answer prompt = teacher's", c["answer_prompt_equals_teacher_generation"],
          "| roundtrip", c["plan_roundtrip_equal"], c["answer_roundtrip_equal"],
          "| tokens", c["n_plan_response"], c["n_answer_response"], "| teacher align", s["teacher_alignment"])
print("\nzero-init logits", r["zero_init"]["logits"])
for k, g in r["zero_init"]["generation"].items():
    print("zero-init generation", k, {x: g[x] for x in ("plan_text_equal", "answer_text_equal", "labels", "matches_saved_prep_student")})
print("kernels (sdpa vs eager)", r["kernels"])
t = r["training"]
print("\nloss by step", {k: [round(x, 4) for x in v] for k, v in t["loss_by_step"].items()})
for st in t["steps"]:
    print("step", st["step"], "skipped", st["skipped"], "scale", st["scale_before"], "->", st["scale_after"],
          "norm", round(st["grad_norm"], 4), "base grads", st["grads"]["n_base_params_with_grad_or_trainable"], st["grads"]["kinds"])
print("mean micro-step", round(t["mean_micro_step_s"], 2), "s | projected training", round(t["projected_training_s"] / 3600, 2), "h | peak", t["peak_gpu_gb"], "GB")
print("\nafter training", r["after_training"])
print("reset", r["reset"])
print("attention after", r["attention_after"])
print("\ngates", json.dumps(r["gates"], indent=1))
print("ALL PASSED", r["all_passed"])
if not r["all_passed"]:
    raise RuntimeError("Implementation check failed. Do not train.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", f"src.{STAGE}", "--phase", "train"])
print("train exit", rc)
t = json.loads(out_file("train_summary.json").read_text(encoding="utf-8"))
for e, b in t["by_epoch"].items():
    print("epoch", e, {k: (round(v, 4) if isinstance(v, float) else v) for k, v in b.items()})
print("params", t["trainable_params"], "| adapter MiB", round(t["adapter_bytes"] / 2**20, 2),
      "| state build s", round(t["state_build_s"]), "| peak GB", t["peak_gpu_gb"],
      "| teacher target generation s (prep)", round(t["teacher_target_generation_s"]))
print("received caches unchanged", t["received_caches_unchanged"])
print("Training loss is on training sheets only; it is not a pilot result.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", f"src.{STAGE}", "--phase", "select"])
print("select exit", rc)
s = json.loads(out_file("selection.json").read_text(encoding="utf-8"))
print("labels by epoch (of 144)", s["labels_by_epoch"], "| unadapted", s["unadapted_labels"], "| selected epoch", s["selected_epoch"])
keys = ("labels_correct", "cases_all_twelve_correct", "ineligible_marked_eligible", "eligible_marked_ineligible", "parse_ok", "plan_truncated")
for e, b in s["by_epoch"].items():
    print("epoch", e, {k: b.get(k) for k in keys})
print("unadapted", {k: s["unadapted"].get(k) for k in keys})

In [ ]:
rc = subprocess.call([sys.executable, "-m", f"src.{STAGE}", "--phase", "gates"])
print("gates exit", rc)
c = json.loads(out_file("dev_construction.json").read_text(encoding="utf-8"))
print("dev construction passed", c["all_passed"])
for k, row in c["cases"].items():
    print(" ", k, "ok", row["ok"], "partner", row["partner"], "differing carriers", row["carriers_differing"],
          "R own/partner", row["n_result_own"], row["n_result_partner"])
d = json.loads(out_file("dev_summary.json").read_text(encoding="utf-8"))
keys = ("labels_correct", "n_labels", "cases_all_twelve_correct", "ineligible_marked_eligible",
        "eligible_marked_ineligible", "parse_ok", "plan_truncated", "plan_lines_missing")
for v in ("base", "a_flip"):
    print(f"\n{v} labels")
    for cond, b in d["labels"][v].items():
        if b.get("n_cases"):
            print(" ", cond, {k: b.get(k) for k in keys})
    print(f"{v} final choice (separate)")
    for cond, b in d["final_choice"][v].items():
        if b.get("n_cases"):
            print(" ", cond, b)
print("\nmean receiver seconds per sheet")
for cond, b in d["costs"].items():
    print(" ", cond, {k: round(x, 3) for k, x in b.items()})
print("\nunadapted under mismatched R (context)", d["context"]["unadapted_mismatched_following"])
print("\nGATES")
for name, g in d["gates"].items():
    print(" ", name, json.dumps(g))
print("\nADVANCE TO UNTOUCHED FAMILIES:", d["advance"])

## Step 4: untouched eval families (only because every dev gate passed)


In [ ]:
for name in ("adapter_epoch1.pt", "selection.json", "dev_summary.json", "train_summary.json"):
    found = [d for d in (WORK / "results_stage_plan4_qadapt", REPO / "results_stage_plan4_qadapt") if (d / name).is_file()]
    print(name, "found" if found else "MISSING")
rc = subprocess.call([sys.executable, "-m", f"src.{STAGE}", "--phase", "freeze"])
print("freeze exit", rc)
if rc != 0:
    raise RuntimeError("Freeze failed. Upload the training run's results_stage_plan4_qadapt folder; do not retrain.")
f = json.loads(out_file("frozen_candidate.json").read_text(encoding="utf-8"))
print("frozen epoch", f["selected_epoch"], "| sha256", f["sha256"], "| dev labels by epoch", f["dev_labels_by_epoch"])


In [ ]:
rc = subprocess.call([sys.executable, "-m", f"src.{STAGE}", "--phase", "eval_write"])
print("eval_write exit", rc)
rows = [json.loads(l) for l in out_file("a_artifacts_eval.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
print("A eval sheets", len(rows), "| truncated", sum(r["written_truncated"] for r in rows),
      "| status correct", sum(r["written_score"]["n_status_correct"] for r in rows), "of", 12 * len(rows))


In [ ]:
rc = subprocess.call([sys.executable, "-m", f"src.{STAGE}", "--phase", "eval_check"])
print("eval_check exit", rc)
c = json.loads(out_file("eval_construction.json").read_text(encoding="utf-8"))
print("all passed", c["all_passed"], "| templates match frozen", c["templates_match_frozen"])
print("A written", c["a_written"], "| A-flip changed carrier written correctly", c["a_flip_changed_carrier_written_correctly"])
for k, r in c["rows"].items():
    if not r["ok"]:
        print("FAILED", k, r)
for k, r in c["rows"].items():
    if r["variant"] == "base":
        d = r["dependence_caches"]
        print(" ", k, "partner", r["partner"], "differing", r["carriers_differing"], "R own/partner", d["n_result_own"], d["n_result_partner"], "ok", d["ok"])


In [ ]:
rc = subprocess.call([sys.executable, "-m", f"src.{STAGE}", "--phase", "eval", "--limit", "1"])
print("eval (first sheet) exit", rc)
rows = [json.loads(l) for l in out_file("eval_evaluations.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
for r in rows:
    print(r["example_id"], r["condition"], "| labels", r.get("plan_n_status_correct"), "| choice", r.get("correct"),
          "| parse", r.get("parse_ok"), "| plan tokens", r.get("n_plan_tokens"), "| kv", r.get("n_kv_positions"),
          "| decode s", round(r.get("b_decode_s") or 0, 2), "| attention", r.get("attention"))
print("\nadapted plan:\n", next(r["plan_text"] for r in rows if r["condition"] == "adapted_correct_r")[:1200])


In [ ]:
rc = subprocess.call([sys.executable, "-m", f"src.{STAGE}", "--phase", "eval"])
print("eval exit", rc)
s = json.loads(out_file("eval_summary.json").read_text(encoding="utf-8"))
print("candidate epoch", s["candidate"]["selected_epoch"], "| rows", s["n_rows"], "| construction", s["construction"]["all_passed"])
keys = ("labels_correct", "n_labels", "cases_all_twelve_correct", "ineligible_marked_eligible",
        "eligible_marked_ineligible", "parse_ok", "plan_truncated", "plan_lines_missing")
for v in ("base", "a_flip"):
    print(f"\n{v} labels")
    for cond, b in s["labels"][v].items():
        if b.get("n_cases"):
            print(" ", cond, {k: b.get(k) for k in keys})
    print(f"{v} final choice")
    for cond, b in s["final_choice"][v].items():
        if b.get("n_cases"):
            print(" ", cond, b)
print("\nmessage dependence (dev gates recomputed on eval, reported only):", s["message_dependence"]["all_hold"])
for k, g in s["message_dependence"]["checks"].items():
    print(" ", k, g)
print("  unadapted mismatched following", s["message_dependence"]["unadapted_mismatched_following"])
print("\npaired by family (adapted minus comparison)")
for name, by_v in s["paired_by_family"].items():
    for v, d in by_v.items():
        lab, a12 = d["labels"], d["all_twelve"]
        print(" ", name, v, "| labels total", lab.get("total_delta"), "CI/family", [round(x, 2) for x in lab.get("ci95_per_case") or []],
              "| all-12 total", a12.get("total_delta"), "CI", [round(x, 2) for x in a12.get("ci95_per_case") or []])
print("\ncosts")
for cond, c in s["costs"]["receiver"].items():
    print(" ", cond, {k: (round(x, 4) if isinstance(x, float) else x) for k, x in c.items()})
print("  adapter overhead", s["costs"]["adapter_overhead"])
print("  adapter", s["costs"]["adapter"], "| one-time", s["costs"]["one_time"])
print(" ", s["costs"]["note"])
